<a href="https://colab.research.google.com/github/elisabethandriivna/Portfolio/blob/main/Lab5_Kaprelova.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, utils, callbacks
from tensorflow.keras.utils import plot_model
from sklearn.metrics import classification_report, confusion_matrix

# ПАРАМЕТРИ
BATCH_SIZE = 64
EPOCHS = 25
IMG_SIZE = 128
NUM_CLASSES = 3
CLASS_MAP = {4:0, 5:1, 7:2} # deer, dog, horse
CLASS_NAMES = ['deer','dog','horse']

# 1. Визначення постановки задачі
# Завантаження CIFAR-10 та відбір класів deer, dog, horse
(x_train_all, y_train_all), (x_test_all, y_test_all) = tf.keras.datasets.cifar10.load_data()

# Функція для фільтрації потрібних класів
def filter_classes(x, y, class_map):
   mask = np.isin(y.flatten(), list(class_map.keys()))
   x_f = x[mask]
   y_f = y[mask].flatten()
   y_mapped = np.array([class_map[int(v)] for v in y_f], dtype=np.int32)
   return x_f, y_mapped

x_train, y_train = filter_classes(x_train_all, y_train_all, CLASS_MAP)
x_test, y_test = filter_classes(x_test_all, y_test_all, CLASS_MAP)

# Перемасштабування зображень та one-hot кодування
x_train = tf.image.resize(x_train, [IMG_SIZE, IMG_SIZE]).numpy().astype('float32') / 127.5 - 1
x_test = tf.image.resize(x_test, [IMG_SIZE, IMG_SIZE]).numpy().astype('float32') / 127.5 - 1

y_train_cat = utils.to_categorical(y_train, NUM_CLASSES)
y_test_cat = utils.to_categorical(y_test, NUM_CLASSES)

print("Train shape:", x_train.shape, y_train_cat.shape)
print("Test shape:", x_test.shape, y_test_cat.shape)

# 2. Побудова моделі
def make_transfer_model(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=NUM_CLASSES):
   inputs = tf.keras.Input(shape=input_shape)

   # Аугментація через Lambda
   x = layers.Lambda(lambda img: tf.image.random_flip_left_right(img))(inputs)
   x = layers.Lambda(lambda img: tf.image.random_brightness(img, 0.2))(x)
   x = layers.Lambda(lambda img: tf.image.random_contrast(img, 0.8, 1.2))(x)

   # Базова модель MobileNetV2
   base = tf.keras.applications.MobileNetV2(
       input_shape=input_shape,
       include_top=False,
       weights='imagenet',
       alpha=1.0
   )

   # Частковий fine-tuning: заморожую все, крім останніх 30 шарів
   for layer in base.layers[:-30]:
       layer.trainable = False
   for layer in base.layers[-30:]:
       layer.trainable = True

   x = base(x, training=True)
   x = layers.GlobalAveragePooling2D()(x)
   x = layers.Dropout(0.3)(x)
   x = layers.Dense(256, activation='relu')(x)
   x = layers.Dropout(0.3)(x)
   outputs = layers.Dense(num_classes, activation='softmax')(x)

   model = tf.keras.Model(inputs, outputs, name='mobilenetv2_cifar3')
   return model, base

model, base_model = make_transfer_model()
model.summary()

# 3. Візуалізація моделі
plot_model(model, to_file='model_plot.png', show_shapes=True, show_layer_names=True)
print("Saved model_plot.png")

# Оптимізатор + scheduler
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)

model.compile(
   optimizer=optimizer,
   loss='categorical_crossentropy',
   metrics=['accuracy']
)

# CALLBACKS
earlystop = callbacks.EarlyStopping(
   monitor='val_loss',
   patience=5,
   restore_best_weights=True
)

lr_reduce = callbacks.ReduceLROnPlateau(
   monitor='val_loss',
   factor=0.3,
   patience=2,
   min_lr=1e-7,
   verbose=1
)

# 4. Навчання моделі
history = model.fit(
   x_train, y_train_cat,
   epochs=EPOCHS,
   batch_size=BATCH_SIZE,
   validation_split=0.2,
   shuffle=True,
   callbacks=[earlystop, lr_reduce],
   verbose=1
)

# Візуалізація процесу навчання
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.title('Loss')
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.title('Accuracy')
plt.legend()
plt.show()

# 5. Визначення показників якості
y_pred = np.argmax(model.predict(x_test), axis=1)

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
plt.imshow(cm, cmap=plt.cm.Blues)
plt.title("Confusion Matrix")
plt.colorbar()
plt.xticks(np.arange(NUM_CLASSES), CLASS_NAMES)
plt.yticks(np.arange(NUM_CLASSES), CLASS_NAMES)

for i in range(NUM_CLASSES):
   for j in range(NUM_CLASSES):
     plt.text(j, i, cm[i,j], ha='center', va='center', color='red')

plt.ylabel('True')
plt.xlabel('Predicted')
plt.show()

   180224/170498071 ━━━━━━━━━━━━━━━━━━━━ 49:18 17us/step

KeyboardInterrupt: 